# FineWeb annotation results

Explore the saved passages and spelling annotations, then review both proposed edits and passages with no proposed edits. **Annotations are model proposals, not ground truth.** A usable result may be incomplete, and an empty result does not prove a passage has no errors.

Run all cells with the project's Python kernel. Everything reads local files; no model calls are made. Change `DATA_DIR`, sample sizes, or the review dictionaries below to explore another run.

In [ ]:
import itertools
import json
import random
import re
from collections import Counter
from html import escape
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import yaml
from IPython.display import HTML, Markdown, display
from wordcloud import STOPWORDS, WordCloud

from speeling_good.preprocessing.types import CorrectionType

plt.rcParams.update(
    {"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False}
)
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
DATA_DIR = ROOT / "data" / "fineweb"
SEED = 42
POSITIVE_SAMPLE_SIZE = 12
EMPTY_SAMPLE_SIZE = 12
TYPE_SAMPLE_SIZE = 5


def table(headers, values):
    head = "".join(f"<th>{escape(str(x))}</th>" for x in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(x))}</td>" for x in row) + "</tr>"
        for row in values
    )
    display(
        HTML(
            f"<div style='overflow:auto;max-height:550px'><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"
        )
    )


def ratio(n, d):
    return f"{n:,} / {d:,} ({n / d:.1%})" if d else "N/A (zero denominator)"


def words(text):
    # A transparent approximation, not the model tokenizer.
    return re.findall(r"[^\W\d_]+(?:['’][^\W\d_]+)*", text, flags=re.UNICODE)


paths = sorted(DATA_DIR.glob("part-*.jsonl"))
if not paths:
    raise FileNotFoundError(f"No completed shards in {DATA_DIR}")
rows: list[dict[str, Any]] = []
for path in paths:
    with path.open() as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            raw = json.loads(line)
            text = raw["text"]
            rows.append(
                {
                    "id": f"{path.name}:{line_number}",
                    "shard": path.name,
                    "text": text,
                    "annotations": raw["annotations"],
                    "failures": raw.get("failures", []),
                    "chars": len(text),
                    "words": len(words(text)),
                }
            )
by_id = {r["id"]: r for r in rows}
usable = [r for r in rows if r["annotations"] is not None]
positive = [r for r in usable if r["annotations"]]
empty = [r for r in usable if not r["annotations"]]
unavailable = [r for r in rows if r["annotations"] is None]
edits: list[dict[str, Any]] = [
    {**a, "passage_id": r["id"], "id": f"{r['id']}#edit-{i}"}
    for r in usable
    for i, a in enumerate(r["annotations"], 1)
]


def validate_correction_types(edits):
    for edit in edits:
        if edit.get("correction_type") not in set(CorrectionType):
            raise ValueError(
                f"{edit['id']}: missing or invalid correction_type. "
                "Regenerate annotations into a fresh output directory and update DATA_DIR; "
                "resuming old shards does not add correction types."
            )


validate_correction_types(edits)
edit_by_id = {a["id"]: a for a in edits}
config_path = DATA_DIR / "config.yaml"
config = yaml.safe_load(config_path.read_text()) if config_path.exists() else {}
usage_path = DATA_DIR / "usage.json"
usage = (
    json.loads(usage_path.read_text()).get("annotation", {})
    if usage_path.exists()
    else {}
)
display(
    Markdown(
        f"Loaded **{len(rows):,} passages** from **{len(paths)} shards** in `{DATA_DIR}`."
    )
)

## Run context and coverage

The saved configuration describes the **latest invocation**, not necessarily every shard in a resumed run. The prompt below is the policy to use during review. “Has corrections” means at least one saved proposal; “empty” means a usable response containing none. Unavailable results are excluded from annotation prevalence rates. Failure history is reported separately and can accompany a recovered or partial result.

In [ ]:
annotation_config = config.get("annotation", {})
table(
    ["Setting", "Saved value"],
    [
        (k, config.get(k, "unavailable"))
        for k in ("dataset", "subset", "split", "take", "workers", "shard_size")
    ]
    + [(f"annotation.{k}", v) for k, v in annotation_config.items() if k != "prompt"],
)
display(
    HTML(
        "<details><summary>Saved annotation prompt</summary><pre style='white-space:pre-wrap'>"
        + escape(annotation_config.get("prompt", "Unavailable"))
        + "</pre></details>"
    )
)
char_counts = np.array([r["chars"] for r in rows])
word_counts = np.array([r["words"] for r in rows])
table(
    ["Measure", "Value"],
    [
        ("Passages / shards", f"{len(rows):,} / {len(paths):,}"),
        ("Usable annotation results", ratio(len(usable), len(rows))),
        ("With corrections, among usable results", ratio(len(positive), len(usable))),
        ("Empty results, among usable results", ratio(len(empty), len(usable))),
        ("Unavailable results", ratio(len(unavailable), len(rows))),
        (
            "Passages with annotation failure history",
            ratio(sum(bool(r["failures"]) for r in rows), len(rows)),
        ),
        (
            "Saved corrections / distinct pairs",
            f"{len(edits):,} / {len({(a['original'], a['corrected']) for a in edits}):,}",
        ),
        (
            "Exact duplicate texts beyond first occurrence",
            len(rows) - len({r["text"] for r in rows}),
        ),
        (
            "Total characters / approximate words",
            f"{sum(char_counts):,} / {sum(word_counts):,}",
        ),
        (
            "Characters: median / p90 / p99 / max",
            " / ".join(
                f"{v:,.0f}" for v in np.percentile(char_counts, [50, 90, 99, 100])
            ),
        ),
        (
            "Approximate words: median / p90 / p99 / max",
            " / ".join(
                f"{v:,.0f}" for v in np.percentile(word_counts, [50, 90, 99, 100])
            ),
        ),
    ],
)

## Lengths, annotation density, and shard variation

Word counts use Unicode letter sequences with optional internal apostrophes. They are approximate and differ from whitespace counts and model tokens. Densities count **proposed edits**, including multiword spans, rather than verified misspelled words. Length groups show association, not model accuracy. Shards preserve source order and are not independent random samples.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(np.log10(np.maximum(char_counts, 1)), bins=30)
axes[0].set(
    xlabel="log10 characters (zero mapped to one)",
    ylabel="Passages",
    title=f"Passage lengths (n={len(rows):,})",
)
counts = Counter(len(r["annotations"]) for r in usable)
axes[1].bar(list(counts), list(counts.values()))
axes[1].set(
    xlabel="Corrections per usable passage",
    ylabel="Passages",
    title=f"Correction counts (n={len(usable):,})",
)
densities = [1000 * len(r["annotations"]) / r["words"] for r in usable if r["words"]]
axes[2].hist(densities, bins=30)
axes[2].set(
    xlabel="Proposed edits / 1,000 approximate words",
    ylabel="Passages",
    title=f"Edit density (n={len(densities):,})",
)
plt.tight_layout()
plt.show()

length_groups = [
    ("<1k", 0, 1000),
    ("1–3k", 1000, 3000),
    ("3–10k", 3000, 10000),
    ("10k+", 10000, float("inf")),
]
group_stats = []
for label, lo, hi in length_groups:
    group = [r for r in usable if lo <= r["chars"] < hi]
    group_stats.append((label, len(group), sum(bool(r["annotations"]) for r in group)))
fig, ax = plt.subplots()
for i, (label, n, n_positive) in enumerate(group_stats):
    if n:
        ax.bar(i, 100 * n_positive / n, color="tab:blue")
    ax.text(i, 102, f"n={n}", ha="center")
ax.set(
    xticks=range(len(group_stats)),
    xticklabels=[g[0] for g in group_stats],
    ylim=(0, 115),
    xlabel="Passage characters",
    ylabel="Usable passages with corrections (%)",
    title="Annotation presence by length",
)
plt.show()

shard_stats = []
for path in paths:
    group = [r for r in rows if r["shard"] == path.name]
    known = [r for r in group if r["annotations"] is not None]
    shard_stats.append(
        (
            path.name,
            len(group),
            len(known),
            ratio(sum(bool(r["annotations"]) for r in known), len(known)),
            sum(len(r["annotations"]) for r in known),
            sum(bool(r["failures"]) for r in group),
            round(float(np.mean([r["chars"] for r in group]))),
        )
        if group
        else (path.name, 0, 0, "N/A", 0, 0, "N/A")
    )
display(
    table(
        [
            "Shard",
            "Passages",
            "Usable",
            "With corrections / usable",
            "Edits",
            "Failure history",
            "Mean chars",
        ],
        shard_stats,
    )
)

print(f"Mean edits per 1000 words = {np.mean(densities)}")

## Vocabulary and correction patterns

Cloud sizes represent occurrence counts, not quality or importance. The passage cloud lowercases words, excludes WordCloud's English stop words and tokens shorter than three letters, and displays at most 100 words. Correction clouds use **complete saved spans**, preserving case and multiword edits, without stop-word filtering. Unique or rare mistakes may be more useful than the largest cloud terms; the tables retain exact counts.

In [ ]:
vocabulary = Counter(
    w.lower()
    for r in rows
    for w in words(r["text"])
    if len(w) >= 3 and w.lower() not in STOPWORDS
)
originals = Counter(a["original"] for a in edits)
replacements = Counter(a["corrected"] for a in edits)
pairs = Counter((a["original"], a["corrected"]) for a in edits)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, title, frequencies in zip(
    axes,
    ["Passage vocabulary", "Original correction spans", "Replacement spans"],
    [vocabulary, originals, replacements],
    strict=True,
):
    if frequencies:
        cloud = WordCloud(
            width=700,
            height=450,
            background_color="white",
            random_state=SEED,
            max_words=100,
            collocations=False,
        ).generate_from_frequencies(frequencies)
        ax.imshow(cloud, interpolation="bilinear")
    else:
        ax.text(0.5, 0.5, "No terms available", ha="center", transform=ax.transAxes)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
table(
    ["Original", "Replacement", "Occurrences"],
    [(a, b, n) for (a, b), n in pairs.most_common(25)],
)
table(["Passage term (filtered)", "Occurrences"], vocabulary.most_common(20))

## Predicted correction types

Each proposal has one model-predicted category. Labels describe the actual edit; detection remains focused on spelling. These predictions are not verified judgments. Correction shares use all saved proposals as the denominator. Passage counts are distinct within each category and can overlap across categories. All categories appear, including those with no proposals.

Examples are reproducible samples of up to five edits per category, with source context. Change `TYPE_SAMPLE_SIZE` above to inspect more.


In [ ]:
def correction_type_stats(edits):
    return [
        (
            category.value,
            len(group),
            ratio(len(group), len(edits)),
            len({edit["passage_id"] for edit in group}),
        )
        for category in CorrectionType
        for group in [[edit for edit in edits if edit["correction_type"] == category]]
    ]


table(
    ["Correction type", "Corrections", "Share of all corrections", "Distinct passages"],
    correction_type_stats(edits),
)
for category in CorrectionType:
    group = [edit for edit in edits if edit["correction_type"] == category]
    selected = random.Random(SEED).sample(group, min(TYPE_SAMPLE_SIZE, len(group)))
    display(Markdown(f"### {category.value}: examples (n={len(selected)})"))
    if not selected:
        display(Markdown("No corrections of this type."))
        continue
    examples = []
    for edit in selected:
        text = by_id[edit["passage_id"]]["text"]
        context = text[max(0, edit["span_start"] - 100) : edit["span_end"] + 100]
        examples.append((edit["id"], edit["original"], edit["corrected"], context))
    table(["Edit ID", "Original", "Replacement", "Source context"], examples)

## Structural integrity and annotation diagnostics

These checks establish whether stored edits can be located and applied, not whether their replacements are correct. Multiple diagnostic messages may refer to the same request attempt. Failure histories do not reliably distinguish eventual recovery from retained partial output; do not infer retry counts or final completeness from message counts.

In [ ]:
integrity_issues = []
for r in usable:
    spans = []
    for i, a in enumerate(r["annotations"], 1):
        start, end = a["span_start"], a["span_end"]
        identifier = f"{r['id']}#edit-{i}"
        valid = (
            isinstance(start, int)
            and isinstance(end, int)
            and 0 <= start < end <= len(r["text"])
        )
        if not valid:
            integrity_issues.append((identifier, "Invalid offsets"))
        elif r["text"][start:end] != a["original"]:
            integrity_issues.append((identifier, "Source span does not match original"))
        if a["original"] == a["corrected"]:
            integrity_issues.append((identifier, "Unchanged replacement"))
        if valid:
            spans.append((start, end, identifier))
    furthest_end = -1
    for start, end, identifier in sorted(spans):
        if start < furthest_end:
            integrity_issues.append((identifier, "Overlapping span"))
        furthest_end = max(furthest_end, end)
table(
    ["Structural measure", "Count"],
    [("Edits checked", len(edits)), ("Issues", len(integrity_issues))],
)
if integrity_issues:
    table(["Edit identifier", "Issue"], integrity_issues)
failures = [(r["id"], f) for r in rows for f in r["failures"]]
stages = Counter(f["stage"] for _, f in failures)
table(["Failure stage", "Diagnostic messages"], stages.most_common())
table(
    ["Exact diagnostic message", "Occurrences"],
    Counter(f["error"] for _, f in failures).most_common(10),
)
table(
    ["Passage", "Attempt", "Stage", "Diagnostic"],
    [
        (identifier, f["attempt"], f["stage"], f["error"])
        for identifier, f in failures[:10]
    ],
)

## Inspect proposals and look for missed errors

The first two groups are reproducible random samples within their respective strata. Read the full text of empty-result passages to look for missed errors. Targeted groups deliberately overrepresent suspicious patterns and cannot estimate their prevalence or annotation accuracy.

All displayed source text is HTML-escaped. Each proposed edit has an identifier for the review dictionaries below. Highlighting uses valid stored offsets; overlapping or invalid edits should be inspected using the integrity table.

In [ ]:
def show_passage(row):
    annotations = row["annotations"]
    status = (
        "unavailable" if annotations is None else f"{len(annotations)} proposed edits"
    )
    title = f"{row['id']} · {status} · {row['chars']:,} characters · {len(row['failures'])} diagnostics"
    display(Markdown(f"**{title}**"))
    if annotations:
        contexts = []
        for i, a in enumerate(annotations, 1):
            start, end = a["span_start"], a["span_end"]
            context = row["text"][
                max(0, start - 100) : min(len(row["text"]), end + 100)
            ]
            contexts.append(
                (
                    f"{row['id']}#edit-{i}",
                    a["correction_type"],
                    a["original"],
                    a["corrected"],
                    context,
                )
            )
        table(
            ["Edit ID", "Correction type", "Original", "Replacement", "Source context"],
            contexts,
        )
    fragments = []
    cursor = 0
    for a in sorted(annotations or [], key=lambda a: a["span_start"]):
        start, end = a["span_start"], a["span_end"]
        if not (cursor <= start < end <= len(row["text"])):
            continue
        fragments.extend(
            [
                escape(row["text"][cursor:start]),
                "<mark title='"
                + escape(a["corrected"], quote=True)
                + "'>"
                + escape(row["text"][start:end])
                + "</mark>",
            ]
        )
        cursor = end
    fragments.append(escape(row["text"][cursor:]))
    display(
        HTML(
            "<details><summary>Full passage (highlighted proposals; hover for replacement)</summary>"
            "<pre style='white-space:pre-wrap;max-height:650px;overflow:auto'>"
            + "".join(fragments)
            + "</pre></details>"
        )
    )


def sample(group, n, seed=SEED):
    return random.Random(seed).sample(group, min(n, len(group)))


positive_sample = sample(positive, POSITIVE_SAMPLE_SIZE)
empty_sample = sample(empty, EMPTY_SAMPLE_SIZE)
for title, group in [
    ("Random passages with corrections", positive_sample),
    ("Random passages with empty results", empty_sample),
]:
    display(Markdown(f"### {title} (n={len(group)})"))
    for row in group:
        show_passage(row)

In [ ]:
def edit_patterns(a):
    original, corrected = a["original"], a["corrected"]
    result = []
    letters = lambda s: "".join(c for c in s.casefold() if c.isalpha())
    if letters(original) == letters(corrected):
        result.append("Punctuation, case, or spacing only")
    tokens = original.casefold().split()
    if any(left == right for left, right in itertools.pairwise(tokens)) and len(
        corrected.split()
    ) < len(tokens):
        result.append("Possible repeated-word removal")
    if len(original.split()) != len(corrected.split()):
        result.append("Word join/split or token count change")
    if len(original) >= 25 or abs(len(corrected) - len(original)) >= 10:
        result.append("Large span or replacement")
    return result


pattern_edits = {}
for a in edits:
    for pattern in edit_patterns(a):
        pattern_edits.setdefault(pattern, []).append(a)
table(
    ["Review cue (categories overlap)", "Edits", "Share of all proposals"],
    [
        (name, len(group), ratio(len(group), len(edits)))
        for name, group in pattern_edits.items()
    ],
)
for name, group in pattern_edits.items():
    display(Markdown(f"### {name}: targeted examples"))
    selected_ids = list(dict.fromkeys(a["passage_id"] for a in group))
    for identifier in random.Random(SEED).sample(
        selected_ids, min(2, len(selected_ids))
    ):
        show_passage(by_id[identifier])
display(Markdown("### Passages with annotation diagnostics"))
for row in sample([r for r in rows if r["failures"]], 3):
    show_passage(row)
    table(
        ["Attempt", "Stage", "Diagnostic"],
        [(f["attempt"], f["stage"], f["error"]) for f in row["failures"]],
    )

REVIEW_ID = None  # Paste a shard:line identifier here and rerun this cell.
if REVIEW_ID is not None:
    if REVIEW_ID in by_id:
        show_passage(by_id[REVIEW_ID])
    else:
        display(Markdown("Identifier not found in the loaded shards."))

## Record a balanced manual review

Edit the dictionaries and rerun this cell to summarize your judgments. They remain in this notebook and never modify the source dataset. Use the saved prompt when deciding whether an edit belongs in scope.

`EDIT_REVIEW` maps edit IDs to `accept`, `reject`, or `uncertain`. `EMPTY_REVIEW` maps passage IDs to `{ "missed_errors": 0, "notes": "..." }`; enter zero only after reading the full passage. Count independently identified error occurrences, not speculative alternatives. You can also review records outside the random samples, but targeted reviews are biased.

The acceptance rate includes uncertain judgments in its denominator and excludes unreviewed proposals. The empty-passage review reports observed misses, **not recall**. Neither number establishes dataset-wide accuracy.

In [ ]:
EDIT_REVIEW: dict[str, str] = {
    # "part-00000.jsonl:2#edit-1": "accept",
}
EMPTY_REVIEW: dict[str, dict[str, Any]] = {
    # "part-00000.jsonl:1": {"missed_errors": 0, "notes": "Read full passage."},
}

review_errors = []
valid_edit_review = {}
for identifier, label in EDIT_REVIEW.items():
    if identifier not in edit_by_id or label not in {"accept", "reject", "uncertain"}:
        review_errors.append((identifier, "Unknown edit ID or invalid label"))
    else:
        valid_edit_review[identifier] = label
valid_empty_review = {}
for identifier, review in EMPTY_REVIEW.items():
    count = review.get("missed_errors") if isinstance(review, dict) else None
    if identifier not in by_id or by_id[identifier]["annotations"] != []:
        review_errors.append((identifier, "Expected an empty-result passage ID"))
    elif type(count) is not int or count < 0:
        review_errors.append(
            (identifier, "missed_errors must be a nonnegative integer")
        )
    else:
        valid_empty_review[identifier] = review
if review_errors:
    table(["Review ID", "Problem (excluded from summary)"], review_errors)

sample_edit_ids = {
    f"{r['id']}#edit-{i}"
    for r in positive_sample
    for i in range(1, len(r["annotations"]) + 1)
}
empty_sample_ids = {r["id"] for r in empty_sample}
review_summary = []
for name, identifiers in [
    ("Random positive-passage sample", sample_edit_ids),
    ("All reviewed proposals (may include targeted examples)", set(edit_by_id)),
]:
    labels = Counter(
        label
        for identifier, label in valid_edit_review.items()
        if identifier in identifiers
    )
    reviewed = sum(labels.values())
    review_summary.append(
        (
            name,
            reviewed,
            labels["accept"],
            labels["reject"],
            labels["uncertain"],
            ratio(labels["accept"], reviewed),
            ratio(reviewed, len(identifiers)),
        )
    )
table(
    [
        "Review group",
        "Reviewed",
        "Accept",
        "Reject",
        "Uncertain",
        "Accepted / reviewed",
        "Review coverage",
    ],
    review_summary,
)
for name, identifiers in [
    ("Random empty-result sample", empty_sample_ids),
    ("All reviewed empty results", {r["id"] for r in empty}),
]:
    reviewed = [
        review
        for identifier, review in valid_empty_review.items()
        if identifier in identifiers
    ]
    table(
        [
            "Review group",
            "Passages read / available",
            "Passages with missed errors",
            "Missed occurrences",
        ],
        [
            (
                name,
                ratio(len(reviewed), len(identifiers)),
                ratio(sum(r["missed_errors"] > 0 for r in reviewed), len(reviewed)),
                sum(r["missed_errors"] for r in reviewed),
            )
        ],
    )

## Annotation usage and reported cost

Only annotation usage is displayed. Cached tokens are a subset of input tokens; reasoning tokens are a subset of output tokens, so do not add them to totals. Response counts are not passage counts and include retries. Resumed runs can include paid work repeated after the last committed shard, and an interrupted run's snapshot may include requests for passages not yet saved.

Per-passage costs below divide reported annotation cost by saved passages. They are descriptive averages, not billing forecasts. Missing reported cost is unknown, not a known zero charge. No timing or per-passage token measurements are stored.

In [ ]:
if not usage:
    display(Markdown("Annotation usage metadata is unavailable."))
else:
    table(
        ["Annotation usage", "Saved value"],
        [
            (k, usage.get(k, "unavailable"))
            for k in (
                "responses",
                "input_tokens",
                "output_tokens",
                "total_tokens",
                "cached_tokens",
                "reasoning_tokens",
                "cost",
                "responses_without_usage",
                "responses_without_cost",
            )
        ],
    )
    cost = usage.get("cost")
    if cost is not None:
        qualifier = (
            " (reported costs only; some responses lack cost metadata)"
            if usage.get("responses_without_cost")
            else " (reported costs)"
        )
        table(
            ["Cost measure", "Value"],
            [
                ("Annotation total" + qualifier, f"${cost:.6f}"),
                ("Per saved passage", f"${cost / len(rows):.8f}" if rows else "N/A"),
                (
                    "Per 1,000 saved passages",
                    f"${1000 * cost / len(rows):.4f}" if rows else "N/A",
                ),
            ],
        )

## What to take away

Use structural checks to establish data integrity, random review samples to assess label usefulness and missed errors, and targeted examples to discover failure modes. Names, technical terms, dialect, intentional spellings, punctuation changes, grammar edits, and extraction repairs need contextual judgment against the saved policy. The contiguous source prefix is not necessarily representative of all FineWeb.

Before using the labels for training, decide whether the reviewed contamination and observed misses fit your task. More examples or a larger independently reviewed reference set may be needed; model agreement or annotation frequency alone cannot establish quality.

In [ ]:
failure_passages = sum(bool(r["failures"]) for r in rows)
display(
    Markdown(
        f"The saved dataset contains **{len(rows):,} passages** and **{len(edits):,} proposed corrections**. "
        f"**{ratio(len(positive), len(usable))}** usable passages contain proposals; "
        f"**{len(empty):,}** have empty results and **{len(unavailable):,}** have unavailable results. "
        f"There are **{len(integrity_issues):,} structural issues** and **{failure_passages:,} passages with annotation diagnostics**.\n\n"
        f"Manual review currently covers **{len(valid_edit_review):,} edits** and "
        f"**{len(valid_empty_review):,} empty-result passages**. "
        "Use the review summaries above to assess scope adherence and observed misses; unreviewed labels remain unverified."
    )
)